In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion
import pandas as pd


/home/lucab/.conda/envs/recenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
songs = pd.read_pickle("./data/Songs")


In [2]:
vc = VocalAssistant(1)

Detected SSH environment; skipping pyttsx3 initialization.


In [4]:
device = 'cuda'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "model_checkpoint_sampled.pth"
pretrained_model = "facebook/wav2vec2-base"
custom_model, custom_processor = load_trained_model(device, custom_model_name, pretrained_model)

/home/lucab/Recommersion/vocal_assistant/emotion/predict_emotion.py:268: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=

Loaded trained model from checkpoint.


In [7]:
vc.talk("What is your mood today?")
"""while True:
    command, vocal_file = vc.take_command()
    print(command)
    break
""" 
file_path = "happy.wav" 
vocal_file = vc.process_audio_file(file_path)
print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0][:2]
print(audeering)
print("Custom: ")
#custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
custom = predict_emotion(custom_model, device, custom_processor, vocal_file)[0].tolist()
print(custom)
#custom model seems to give the same results: overfitting?
#[0.4807744026184082, 0.5821987390518188, 0.6608558893203735]


Simulating speech: What is your mood today?
Processing the file: happy.wav
Numpy array shape: (71284,)
Recognized speech: i'm so happy today
Audeering: 
[0.7210903  0.72628665]
Custom: 
[0.02223336324095726, 0.09918200969696045]


In [ ]:
import numpy as np
dim_vec = np.array(audeering[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list
#create temp dir?

In [ ]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()


In [7]:
pd.read_pickle("./data/IEMOCAP_useful")

ModuleNotFoundError: No module named 'numpy._core.numeric'